In [1]:
import numpy as np

In [2]:
A = np.array([
    [2, 6],
    [5, 3],
    [-4, 0]
])

right = A.T @ A
left = A @ A.T


In [3]:
def projection(vector, axis):
    return (np.dot(vector, axis) / np.linalg.norm(axis) ** 2) * axis


def get_eigenvalues(matrix):
    m, n = matrix.shape

    eigenvalues = matrix.copy()
    eigenvectors = np.eye(m, n)

    for iter in range(max_iter):
        prev = eigenvalues.copy()

        a = np.array([eigenvalues[:, i] for i in range(n)])
        Q = [a[0]]
        for i in range(1, n):
            b_tmp = 0
            a_i = a[i]

            for b_i in Q:
                b_tmp += projection(a_i, b_i)

            Q += [a_i - b_tmp]

        Q = np.array([b_i / np.linalg.norm(b_i) for b_i in Q]).T

        R = Q.T @ eigenvalues
        eigenvalues = R @ Q
        eigenvectors @= Q

        if np.linalg.norm(eigenvalues.diagonal() - prev.diagonal()) < eps:
            break

    return np.sort(eigenvalues.diagonal())[::-1], eigenvectors


def get_eigenvectors(eigenvalues, eigenvectors):
    return np.array([eigenvalues[i] * eigenvectors[:, i] for i in range(len(eigenvalues))])


def get_singularvalues(eigenvalues):
    return np.diag(eigenvalues ** .5)


eps = 1e-8
max_iter = 10 ** 5

print(A)
print()

E, V = get_eigenvalues(right)
V = get_eigenvectors(E, V)

E_U, U = get_eigenvalues(left)
U = get_eigenvectors(np.append(E, [0]), U)

E = get_singularvalues(E)
print(U, E, V, sep="\n\n")

approx = U[:, :-1] @ E @ V

print()
print(approx)

print()
print(A - approx)

print()
print(np.linalg.norm(A - approx))

[[ 2  6]
 [ 5  3]
 [-4  0]]

[[-7.59403903e+05  6.91016763e+06 -5.92192152e+06]
 [-1.20862531e+03  2.64083471e+02 -7.34702907e+02]
 [ 0.00000000e+00 -0.00000000e+00  0.00000000e+00]]

[[8.48528137 0.        ]
 [0.         4.24264069]]

[[ 50.9117368   50.91163969]
 [-12.72790992  12.7279342 ]]

[[-7.01211495e+08  4.50872350e+07]
 [-5.36387127e+05 -5.07865116e+05]
 [ 0.00000000e+00  0.00000000e+00]]

[[ 7.01211497e+08 -4.50872290e+07]
 [ 5.36392127e+05  5.07868116e+05]
 [-4.00000000e+00  0.00000000e+00]]

702659923.3235044


In [4]:
def power_iteration(A, max_iter):
    x = np.random.randint(10, size=A.shape[1])

    prev = 0
    for _ in range(max_iter):
        x = A @ x / np.linalg.norm(A @ x)
        lam = (x.T @ A @ x) / (x.T @ x)

        if abs(lam - prev) < eps:
            break

    return lam, x


max_iter = 10 ** 4
eps = 1e-4

print(*power_iteration(left, max_iter))
print(*power_iteration(right, max_iter))

72.0 [ 0.66666667  0.66666667 -0.33333333]
72.0 [0.70710678 0.70710678]


In [5]:
def randomized_svd(A, k=6, p=10, q=2, random_state=None):
    m, n = A.shape

    if random_state:
        np.random.seed(random_state)

    Omega = np.random.randn(n, k + p)

    Y = A @ Omega

    Q, _ = np.linalg.qr(Y)

    for _ in range(q):
        Y = A.T @ Q
        Q, _ = np.linalg.qr(Y)
        Y = A @ Q
        Q, _ = np.linalg.qr(Y)

    B = Q.T @ A

    U, E, V = np.linalg.svd(B, full_matrices=False)

    U = Q @ U

    U = U[:, :k]
    E = E[:k]
    V = V[:k, :]

    return U, E, V



U, E, V = randomized_svd(A, random_state=42)
E = np.diag(E)

approx = U @ E @ V

print(A)
print()

print(U, E, V, sep="\n\n")

print()
print(approx)
print()

print(A - approx)
print()

print(np.linalg.norm(A - approx))

[[ 2  6]
 [ 5  3]
 [-4  0]]

[[ 0.66666667  0.66666667]
 [ 0.66666667 -0.33333333]
 [-0.33333333  0.66666667]]

[[8.48528137 0.        ]
 [0.         4.24264069]]

[[ 0.70710678  0.70710678]
 [-0.70710678  0.70710678]]

[[ 2.00000000e+00  6.00000000e+00]
 [ 5.00000000e+00  3.00000000e+00]
 [-4.00000000e+00 -6.14746073e-16]]

[[ 2.22044605e-16  0.00000000e+00]
 [-8.88178420e-16  4.44089210e-16]
 [ 8.88178420e-16  6.14746073e-16]]

1.4839654908644266e-15
